<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Exorust/TorchLeet/blob/main/llm/Rotary-Positional-Embedding/rope-q8-Question.ipynb)

Check your work: `!pip install torchleet` then `from torchleet import check; check("rotary-positional-embedding", ...)`


# Implement RoPE from Scratch
# 🔁 Rotary Positional Embeddings in PyTorch

### 🧠 Problem Statement
Self-attention has no built-in notion of order, so positional information has to be injected. Instead of *adding* a position vector to the token embedding (as sinusoidal embeddings do), **Rotary Positional Embeddings (RoPE)** *rotate* the query and key vectors by an angle that grows with the token's position. RoPE is used by Llama, Mistral, Qwen, Gemma and most modern LLMs.

Split the `head_dim` features of a query/key into `head_dim / 2` pairs. At position `m`, pair `i` is rotated by the angle `m · θ_i`, with

`θ_i = base^(-2i / head_dim)`, `i = 0, 1, ..., head_dim/2 - 1`, `base = 10000`.

Because rotations compose, `⟨R_m q, R_n k⟩ = ⟨q, R_{n-m} k⟩`: the attention score depends only on the **relative** offset `m − n`, and RoPE adds no parameters. (On its own RoPE does not extrapolate well past the training length; position interpolation, NTK and YaRN scaling exist to fix that.)

Which features form a pair is a convention. The original RoFormer and GPT-J pair adjacent features `(2i, 2i+1)`; GPT-NeoX and Hugging Face's Llama pair feature `i` with `i + head_dim/2`. Both are correct as long as the cos/sin layout matches the pairing. Use the second one (the `rotate_half` convention) here.

---

### ✅ Requirements

1. **`Rotary(dim, base=10000)`** — an `nn.Module` for one attention head of size `dim`:
   - Precompute the inverse frequencies `θ_i` and register them as a buffer.
   - `forward(x, seq_dim=-2)` returns `(cos, sin)`, each of shape `(seq_len, dim)`, for positions `0 .. seq_len-1`, with the angles laid out to match `rotate_half` (`cat(freqs, freqs)`).
   - Cache the tables as buffers and only recompute when a longer sequence arrives.

2. **`rotate_half(x)`** — split the last dimension into halves `(x1, x2)` and return `cat(-x2, x1)`.

3. **`apply_rotary_pos_emb(q, k, cos, sin)`** — rotate `q` and `k` of shape `(batch, heads, seq_len, head_dim)` and return `(q_rot, k_rot)` with the same shapes. `cos`/`sin` of shape `(seq_len, head_dim)` must broadcast against them.

4. **Verify**
   - Shapes are unchanged.
   - Norms are preserved (it is a rotation) and position 0 is not rotated.
   - The score between a fixed `q` at position `m` and a fixed `k` at position `n` depends only on `m − n`.

---

### 📏 Constraints

- ✅ Use only PyTorch — no Hugging Face or other positional-embedding modules.
- ✅ `head_dim` must be even (features are rotated in pairs).
- ✅ Support any sequence length; apply RoPE per head, to `q` and `k` only (never to `v`), before the `q @ kᵀ` scores.

---

<details>
  <summary>💡 Hint</summary>

  - `torch.outer(t, inv_freq)` (or `torch.einsum("i,j->ij", t, inv_freq)`) gives the angle for every (position, pair).
  - Cache the cosine and sine tables in the `Rotary` module with `self.register_buffer(..., persistent=False)`.
  - `rotate_half(x)` splits `x` into two halves and returns `[-x2, x1]`.
  - The rotation is `(q * cos) + (rotate_half(q) * sin)`, and the same for `k`.
</details>

In [ ]:
import torch
import torch.nn as nn

In [ ]:
torch.manual_seed(42)

In [ ]:
class Rotary(nn.Module):
    def __init__(self, dim, base=10000.0):
        super().__init__()
        # TODO: inverse frequencies theta_i = base^(-2i/dim), i = 0 .. dim/2 - 1, as a buffer
        ...

    def forward(self, x, seq_dim=-2):
        """Return (cos, sin), each of shape (seq_len, dim), for positions 0 .. seq_len-1."""
        # TODO: angles m * theta_i, laid out as cat(freqs, freqs) to match rotate_half;
        # cache the tables as buffers and reuse them for shorter sequences
        ...


def rotate_half(x):
    # TODO: split the last dimension into halves (x1, x2) and return cat(-x2, x1)
    ...


def apply_rotary_pos_emb(q, k, cos, sin):
    # TODO: rotate q and k, each (batch, heads, seq_len, head_dim); return (q_rot, k_rot)
    ...

In [ ]:
# Apply RoPE to per-head queries and keys and check the properties that define it.
batch_size, num_heads, seq_len, head_dim = 2, 4, 50, 64
q = torch.randn(batch_size, num_heads, seq_len, head_dim)
k = torch.randn(batch_size, num_heads, seq_len, head_dim)

rotary = Rotary(head_dim)
cos, sin = rotary(q)                      # each (seq_len, head_dim)
q_rot, k_rot = apply_rotary_pos_emb(q, k, cos, sin)
print(q_rot.shape, k_rot.shape)           # (2, 4, 50, 64) (2, 4, 50, 64)
assert q_rot.shape == q.shape and k_rot.shape == k.shape

# 1. A rotation preserves norms, and position 0 (angle 0) is left alone.
assert torch.allclose(q_rot.norm(dim=-1), q.norm(dim=-1), atol=1e-4)
assert torch.allclose(q_rot[:, :, 0], q[:, :, 0], atol=1e-6)

# 2. Relative position: the same q at position m and the same k at position n
#    score <R_m q, R_n k>, which must depend only on m - n.
q_same = torch.randn(head_dim).expand(1, 1, seq_len, head_dim)
k_same = torch.randn(head_dim).expand(1, 1, seq_len, head_dim)
rq, rk = apply_rotary_pos_emb(q_same, k_same, cos, sin)
scores = rq[0, 0] @ rk[0, 0].T            # scores[m, n]
assert torch.allclose(scores[1:, 1:], scores[:-1, :-1], atol=1e-4)  # shift both by 1
print("norms preserved, scores depend only on m - n")